# 10 - End-to-End Pipeline Showcase

Demonstration notebook for the video presentation. It loads the four trained models saved by notebooks 04-07 and runs them over the held-out test split to show:

1. how each model classifies the traffic jam level of a single random 15-second interval, and
2. predicted vs actual `vehicle_count` for every model across the whole test split.

Nothing is retrained here - this reuses `models/*.pkl` and `models/*.pt` together with the same preprocessing pipeline used during training, so the numbers line up with `results/model_comparison.csv`.

Output: figures saved to `results/figures/`

**Split:** S01, S03, S04 = train - S02 = validation - S05 = test

In [ ]:
import random
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from pytorch_tcn import TCN
from sklearn.metrics import accuracy_score

sys.path.append("..")

from src.data_utils import (
    add_delta_target,
    drop_incomplete_lag_rows,
    lag1_raw_column,
    load_density_data,
    scale_features,
    split_by_column,
    target_column,
)
from src.eval_metrics import regression_metrics
from src.sequence_utils import create_sequences, flatten_for_baseline

In [ ]:
SEED = 24
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

processed_dir = Path("../data/processed")
input_path = processed_dir / "traffic_density.csv"

model_dir = Path("../models")
figures_dir = Path("../results/figures")
figures_dir.mkdir(parents=True, exist_ok=True)

# same quantile thresholds and category order used in notebooks 03 and 09
density_quantiles = [0.3, 0.6]
category_order = ["low", "moderate", "high"]

interval_length_sec = 15

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 1. Rebuild the test split through the training pipeline

In [ ]:
density = load_density_data(input_path)
density = drop_incomplete_lag_rows(density)
density = add_delta_target(density)

# unique row id so every generated sequence can be traced back to the interval it predicts
uid_column = "row_uid"
density[uid_column] = np.arange(len(density))

train_df, val_df, test_df = split_by_column(density)
train_df, val_df, test_df, scaler = scale_features(train_df, val_df, test_df)

print(f"train: {len(train_df)} rows, val: {len(val_df)} rows, test: {len(test_df)} rows")

In [ ]:
low_threshold, high_threshold = train_df[target_column].quantile(density_quantiles)


def categorize_density(value, low_threshold, high_threshold):
    if value <= low_threshold:
        return "low"
    elif value <= high_threshold:
        return "moderate"
    else:
        return "high"


print(f"low / moderate boundary: {low_threshold:.4f}")
print(f"moderate / high boundary: {high_threshold:.4f}")

## 2. Model definitions

These classes are copied from notebooks 05, 06 and 07 - the saved state dicts only load into the identical architecture.

In [ ]:
class lstm_model(nn.Module):
    def __init__(self, n_features, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            n_features, hidden_size, num_layers, batch_first=True, dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out).squeeze(-1)


class tcn_model(nn.Module):
    def __init__(self, n_features, hidden_size, num_layers, kernel_size, dropout, dilations):
        super().__init__()
        num_channels = [hidden_size] * num_layers
        self.tcn = TCN(
            num_inputs=n_features,
            num_channels=num_channels,
            kernel_size=kernel_size,
            dilations=dilations,
            dropout=dropout,
            causal=True,
            input_shape="NLC",
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out = self.tcn(x)
        out = out[:, -1, :]
        return self.fc(out).squeeze(-1)


class tcn_lstm_hybrid(nn.Module):
    def __init__(self, n_features, hidden_size, tcn_layers, kernel_size, dropout, lstm_layers, dilations):
        super().__init__()
        tcn_channels = [hidden_size] * tcn_layers
        self.tcn = TCN(
            num_inputs=n_features,
            num_channels=tcn_channels,
            kernel_size=kernel_size,
            dilations=dilations,
            dropout=dropout,
            causal=True,
            input_shape="NLC",
        )
        self.lstm = nn.LSTM(hidden_size, hidden_size, lstm_layers, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out = self.tcn(x)
        out, _ = self.lstm(out)
        out = out[:, -1, :]
        out = self.dropout(out)
        return self.fc(out).squeeze(-1)

In [ ]:
# architecture hyperparameters, copied from notebooks 05-07
lstm_num_layers = 2
lstm_dropout = 0.2

tcn_layers = 3
tcn_dilations = [1, 2, 4]
tcn_kernel_size = 2
tcn_dropout = 0.1

hybrid_lstm_layers = 1

# window_size / hidden_size are the configs each notebook's sweep selected.
# target_mode "delta" means the model predicts the change from lag_1, so the raw count is
# recovered by adding lag_1 back (notebooks 05 and 07). "raw" means it predicts the count directly.
model_configs = {
    "linear_regression": dict(kind="sklearn", path=model_dir / "linear_regression.pkl", window_size=2, hidden_size=None, target_mode="raw"),
    "lstm": dict(kind="torch", path=model_dir / "lstm.pt", window_size=2, hidden_size=32, target_mode="delta"),
    "tcn": dict(kind="torch", path=model_dir / "tcn.pt", window_size=2, hidden_size=32, target_mode="raw"),
    "tcn_lstm_hybrid": dict(kind="torch", path=model_dir / "tcn_lstm_hybrid.pt", window_size=2, hidden_size=16, target_mode="delta"),
}

model_names = list(model_configs)


def build_model(model_name, n_features):
    hidden_size = model_configs[model_name]["hidden_size"]

    if model_name == "lstm":
        return lstm_model(n_features, hidden_size, lstm_num_layers, lstm_dropout)
    if model_name == "tcn":
        return tcn_model(n_features, hidden_size, tcn_layers, tcn_kernel_size, tcn_dropout, tcn_dilations)
    if model_name == "tcn_lstm_hybrid":
        return tcn_lstm_hybrid(n_features, hidden_size, tcn_layers, tcn_kernel_size, tcn_dropout, hybrid_lstm_layers, tcn_dilations)

    raise ValueError(f"no torch architecture defined for {model_name}")

## 3. Run every saved model over the test split

In [ ]:
prediction_frames = []

for model_name, config in model_configs.items():
    window_size = config["window_size"]

    x_test, y_test = create_sequences(test_df, window_size, target=target_column)
    _, y_test_lag1 = create_sequences(test_df, window_size, target=lag1_raw_column)
    _, y_test_uid = create_sequences(test_df, window_size, target=uid_column)

    if config["kind"] == "sklearn":
        loaded_model = joblib.load(config["path"])
        preds = loaded_model.predict(flatten_for_baseline(x_test))
    else:
        loaded_model = build_model(model_name, x_test.shape[2]).to(device)
        loaded_model.load_state_dict(torch.load(config["path"], map_location=device))
        loaded_model.eval()

        x_test_tensor = torch.tensor(x_test, dtype=torch.float32).to(device)
        with torch.no_grad():
            preds = loaded_model(x_test_tensor).cpu().numpy()

    if config["target_mode"] == "delta":
        preds = preds + y_test_lag1

    prediction_frames.append(pd.DataFrame({
        "model_name": model_name,
        uid_column: y_test_uid.astype(int),
        "actual": y_test,
        "predicted": preds,
    }))

    print(f"{model_name}: {len(y_test)} intervals predicted")

showcase_df = pd.concat(prediction_frames, ignore_index=True)

In [ ]:
meta_columns = ["scene_id", "camera_id", "interval_id", "interval_start_sec"]
meta = test_df.set_index(uid_column)[meta_columns]

showcase_df = showcase_df.join(meta, on=uid_column)
showcase_df["actual_category"] = showcase_df["actual"].apply(categorize_density, args=(low_threshold, high_threshold))
showcase_df["predicted_category"] = showcase_df["predicted"].apply(categorize_density, args=(low_threshold, high_threshold))
showcase_df["correct_category"] = showcase_df["actual_category"] == showcase_df["predicted_category"]

showcase_df.head()

## 4. Demo - jam classification for one random 15-second interval

Every row of the test split is one 15-second interval from one camera. Change `SAMPLE_SEED` to draw a different interval.

In [ ]:
SAMPLE_SEED = 7

# only intervals every model produced a prediction for (all four share window_size = 2)
shared_uids = sorted(set.intersection(*[
    set(showcase_df.loc[showcase_df["model_name"] == name, uid_column]) for name in model_names
]))

sample_uid = random.Random(SAMPLE_SEED).choice(shared_uids)
sample_rows = showcase_df[showcase_df[uid_column] == sample_uid].set_index("model_name")
sample_meta = meta.loc[sample_uid]

interval_start = int(sample_meta["interval_start_sec"])
sample_actual = float(sample_rows["actual"].iloc[0])
sample_actual_category = sample_rows["actual_category"].iloc[0]

print(f"scene {sample_meta['scene_id']} - camera {sample_meta['camera_id']} - interval {int(sample_meta['interval_id'])}")
print(f"time window: {interval_start}s to {interval_start + interval_length_sec}s")
print(f"actual vehicle count: {sample_actual:.4f}  ->  {sample_actual_category} density")

In [ ]:
# the intervals the models actually saw as input for this prediction (the preceding window)
window_size = model_configs["tcn"]["window_size"]
input_uids = list(range(sample_uid - window_size, sample_uid))

input_view_columns = ["scene_id", "camera_id", "interval_start_sec", "vehicle_count", "lag_1", "rolling_mean", "rate_of_change"]
input_view = density[density[uid_column].isin(input_uids)][input_view_columns]

print("model input - the previous intervals from the same camera (unscaled values):")
input_view.round(3)

In [ ]:
sample_table = sample_rows.loc[model_names, ["predicted", "predicted_category", "actual", "actual_category", "correct_category"]].copy()
sample_table["error"] = sample_table["predicted"] - sample_table["actual"]

sample_table[["actual", "actual_category", "predicted", "error", "predicted_category", "correct_category"]].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

predicted_values = [sample_rows.loc[name, "predicted"] for name in model_names]
bar_colors = ["tab:green" if sample_rows.loc[name, "correct_category"] else "tab:red" for name in model_names]

bars = ax.bar(model_names, predicted_values, color=bar_colors)
ax.axhline(sample_actual, linestyle="--", color="black", label=f"actual ({sample_actual:.2f})")
ax.axhline(low_threshold, linestyle=":", color="grey", label=f"low / moderate ({low_threshold:.2f})")
ax.axhline(high_threshold, linestyle=":", color="dimgrey", label=f"moderate / high ({high_threshold:.2f})")

for bar, model_name in zip(bars, model_names):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        sample_rows.loc[model_name, "predicted_category"],
        ha="center",
        va="bottom",
    )

ax.set_ylabel("vehicle count")
ax.set_title(
    f"{sample_meta['scene_id']} {sample_meta['camera_id']} - {interval_start}s to {interval_start + interval_length_sec}s"
    f" (actual: {sample_actual_category})"
)
ax.set_ylim(0, max(predicted_values + [sample_actual, high_threshold]) * 1.15)
ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.xticks(rotation=15)

fig.tight_layout()
fig.savefig(figures_dir / "pipeline_showcase_sample_interval.png", dpi=300)
plt.show()

## 5. Predicted vs actual vehicle count, all models

In [ ]:
fig, axes = plt.subplots(1, len(model_names), figsize=(4.5 * len(model_names), 4), sharex=True, sharey=True)

for ax, model_name in zip(axes, model_names):
    model_df = showcase_df[showcase_df["model_name"] == model_name]

    ax.scatter(model_df["actual"], model_df["predicted"], s=12, alpha=0.4)

    upper = max(model_df["actual"].max(), model_df["predicted"].max()) * 1.05
    lower = min(0.0, model_df["predicted"].min())
    ax.plot([lower, upper], [lower, upper], linestyle="--", color="black", linewidth=1)

    ax.set_xlabel("actual")
    ax.set_ylabel("predicted")
    ax.set_title(model_name)

fig.suptitle("Predicted vs actual vehicle count (test split)")
fig.tight_layout()
fig.savefig(figures_dir / "pipeline_showcase_predicted_vs_actual.png", dpi=300)
plt.show()

In [ ]:
# same camera as the sampled interval, plotted over time so the density curve is visible
camera_df = showcase_df[
    (showcase_df["scene_id"] == sample_meta["scene_id"])
    & (showcase_df["camera_id"] == sample_meta["camera_id"])
]

fig, ax = plt.subplots(figsize=(11, 4))

actual_series = camera_df[camera_df["model_name"] == model_names[0]].sort_values("interval_start_sec")
ax.plot(actual_series["interval_start_sec"], actual_series["actual"], color="black", linewidth=2, label="actual")

for model_name in model_names:
    model_series = camera_df[camera_df["model_name"] == model_name].sort_values("interval_start_sec")
    ax.plot(model_series["interval_start_sec"], model_series["predicted"], linewidth=1.2, alpha=0.85, label=model_name)

ax.axvline(interval_start, color="tab:orange", linestyle="--", linewidth=1, label="sampled interval")
ax.set_xlabel("interval start (seconds)")
ax.set_ylabel("vehicle count")
ax.set_title(f"Actual vs predicted vehicle count - {sample_meta['scene_id']} {sample_meta['camera_id']}")
ax.legend(ncol=3, fontsize=8)

fig.tight_layout()
fig.savefig(figures_dir / "pipeline_showcase_timeseries.png", dpi=300)
plt.show()

## 6. Summary

In [ ]:
summary_rows = []

for model_name in model_names:
    model_df = showcase_df[showcase_df["model_name"] == model_name]
    metrics = regression_metrics(model_df["actual"].to_numpy(), model_df["predicted"].to_numpy())

    summary_rows.append(dict(
        model_name=model_name,
        **metrics,
        jam_accuracy=accuracy_score(model_df["actual_category"], model_df["predicted_category"]),
        sample_interval_correct=bool(sample_rows.loc[model_name, "correct_category"]),
    ))

summary_df = pd.DataFrame(summary_rows).sort_values("rmse").reset_index(drop=True)
summary_df.round(4)